In [ ]:
### YOLOv8 Colab 학습 전체 코드
# 1. Ultralytics 설치
!pip install ultralytics

In [ ]:
# YOLO-Seg 기반 Grid Cell Segmentation + Mask Upscale + Pixel Area 계산 파이프라인

# (5) YOLO-Seg inference
# (6) mask 원본크기로 inverse-resize
# (7) pixel area 계산 및 저장

# 사용자 입력 필요:
# - GRID_FOLDER: grid split 이미지 저장된 폴더
# - YOLO_MODEL: yolov8-seg 모델 경로
# - OUTPUT_CSV: 결과 저장 경로

import os
import cv2
import numpy as np
from glob import glob
import torch
from ultralytics import YOLO
import pandas as pd
from tqdm import tqdm

# =============================
# 0. 기본 경로 설정
# =============================
GRID_FOLDER = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/3. ROI_LETTUCE/bed_grid_split"  # grid 결과 폴더
YOLO_MODEL_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/Practice/251011 양상추 seg 연습(2)/result/lettuce_yolov8n4/weights/best.pt"  # 기존 세그모델 경로
OUTPUT_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/251207_lettuce_segmentation_area.csv"

In [ ]:

# =============================
# 1. YOLO-Seg 모델 로드
# =============================
model = YOLO(YOLO_MODEL_PATH)
print("YOLO-Seg 모델 로딩 완료")


YOLO-Seg 모델 로딩 완료


In [ ]:

# =============================
# 2. pixel area 계산 함수
# =============================
def calculate_area(mask):
    """
    mask: 0/1 binary mask
    return: pixel area (int)
    """
    return int(mask.sum())

# =============================
# 3. inverse-resize (원본 크기 복원)
# =============================
def upscale_mask(mask, original_h, original_w):
    """
    YOLO-Seg 출력 mask(256x256 등)를 원래 이미지 크기로 복원
    """
    up = cv2.resize(mask, (original_w, original_h), interpolation=cv2.INTER_NEAREST)
    return up

# =============================
# 4. 전체 Grid Cell 대상 Segmentation (chunk 처리 + batch=24)
# =============================
import time

records = []
cell_images = glob(os.path.join(GRID_FOLDER, "**", "*.png"), recursive=True)
cell_images = sorted(cell_images)

total = len(cell_images)
print(f"총 grid cell 이미지 개수: {total}")

# 사용자 설정
BATCH_SIZE = 24       # T4에서 안전하며 빠른 batch
CHUNK_SIZE = 3000     # 3000장 단위로 잘라서 처리

start_time = time.time()

from tqdm import tqdm

def process_chunk(chunk_paths, records):
    """YOLO-Seg 배치 추론을 chunk 단위로 수행"""
    results = model.predict(
        source=chunk_paths,
        imgsz=256,
        verbose=False,
        batch=BATCH_SIZE,
    )

    for img_path, res in zip(chunk_paths, results):
        img = cv2.imread(img_path)
        if img is None:
            continue

        orig_h, orig_w = img.shape[:2]

        if res.masks is None or len(res.masks) == 0:
            area_px = 0
            records.append([img_path, orig_h, orig_w, area_px])
            continue

        masks = res.masks.data.cpu().numpy()
        areas = [m.sum() for m in masks]
        max_idx = int(np.argmax(areas))
        seg_mask = masks[max_idx]

        restored_mask = upscale_mask(seg_mask, orig_h, orig_w)
        area_px = calculate_area(restored_mask)

        records.append([img_path, orig_h, orig_w, area_px])


# chunk 나누기
num_chunks = (total + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"총 {num_chunks}개의 chunk로 분할하여 처리합니다 (chunk 당 {CHUNK_SIZE}장)")

for i in range(num_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total)
    chunk_paths = cell_images[start_idx:end_idx]

    print(f"\n=== Chunk {i+1}/{num_chunks} 처리중 ({len(chunk_paths)}장) ===")
    process_chunk(chunk_paths, records)

elapsed = time.time() - start_time
print(f"YOLO-Seg 전체 완료: 총 {elapsed/60:.2f}분 소요")

# =============================
# 5. CSV 저장
# =============================
df = pd.DataFrame(records, columns=["image_path", "orig_h", "orig_w", "area_px"])
df.to_csv(OUTPUT_CSV, index=False)
print("저장 완료:", OUTPUT_CSV)

총 grid cell 이미지 개수: 18768
총 7개의 chunk로 분할하여 처리합니다 (chunk 당 3000장)

=== Chunk 1/7 처리중 (3000장) ===
WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs


=== Chunk 2/7 처리중 (3000장) ===
WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    fo

## 4. 계산된 픽셀 기반 면적 환산하기

In [ ]:
# =============================================================
# Pixel Scale Map + Segmentation Area 매칭 후 mm^2 면적 환산
# -------------------------------------------------------------
# - scale_map CSV: image_name, mm_per_px 등 포함
# - seg_area CSV: image_path, orig_h, orig_w, area_px
# - parent_name(.jpg 없음) 을 기준으로 매칭
# =============================================================

import os
import pandas as pd
import numpy as np

# -------------------------------------------------------------
# 0. 파일 경로 설정
# -------------------------------------------------------------
SCALE_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/251207_pixel_scale_map.csv"
SEG_AREA_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/251207_lettuce_segmentation_area.csv"
OUTPUT_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/251207_segmentation_area_mm2.csv"

# -------------------------------------------------------------
# 1. 데이터 불러오기
# -------------------------------------------------------------
scale_df = pd.read_csv(SCALE_CSV)
seg_df = pd.read_csv(SEG_AREA_CSV)

print("로드 완료:")
print(" - scale map:", scale_df.shape)
print(" - segmentation area:", seg_df.shape)

로드 완료:
 - scale map: (1628, 5)
 - segmentation area: (18768, 6)


In [ ]:


# -------------------------------------------------------------
# 2. parent_name 생성 (seg_df의 image_path에서 폴더명 추출)
# -------------------------------------------------------------
# 예: .../bed00_20251204_102819_cam0/bed00_20251204_102819_cam0_00.png
# → parent_name = bed00_20251204_102819_cam0

seg_df["parent_name"] = seg_df["image_path"].apply(lambda x: os.path.basename(os.path.dirname(x)))

# -------------------------------------------------------------
# 3. scale map의 image_name 정규화
# -------------------------------------------------------------
# scale map의 image_name에는 .jpg가 붙어 있음 → 제거
# 예: bed00_20251204_102819_cam0.jpg → bed00_20251204_102819_cam0

scale_df["parent_name"] = scale_df["image_name"].apply(lambda x: os.path.splitext(str(x))[0])

# -------------------------------------------------------------
# 4. 매칭 (left join)
# -------------------------------------------------------------
merged = seg_df.merge(scale_df, on="parent_name", how="left", suffixes=("_seg", "_scale"))

print("매칭 후 shape:", merged.shape)


매칭 후 shape: (18840, 11)


In [ ]:


# -------------------------------------------------------------
# 5. mm^2 면적 계산
# -------------------------------------------------------------
# px_per_mm_x, px_per_mm_y를 이용해 면적 환산
# area_mm2 = area_px / (px_per_mm_x * px_per_mm_y)


required_cols = ["area_px", "px_per_mm_x", "px_per_mm_y"]
missing_cols = [c for c in required_cols if c not in merged.columns]
if missing_cols:
  raise KeyError(f"다음 컬럼이 존재하지 않습니다: {missing_cols}")


# 0 또는 NaN 방지
valid_mask = (merged["px_per_mm_x"] > 0) & (merged["px_per_mm_y"] > 0)
merged.loc[valid_mask, "area_mm2"] = merged.loc[valid_mask, "area_px"] / (
merged.loc[valid_mask, "px_per_mm_x"] * merged.loc[valid_mask, "px_per_mm_y"]
)
merged.loc[~valid_mask, "area_mm2"] = np.nan


# -------------------------------------------------------------
# 6. 결과 저장
# -------------------------------------------------------------
merged.to_csv(OUTPUT_CSV, index=False)
print("저장 완료:", OUTPUT_CSV)


# -------------------------------------------------------------
# 7. 결측치 체크 (scale map과 매칭 실패한 경우)
# -------------------------------------------------------------
missing = merged[merged["px_per_mm_x"].isna()]
print("scale map 매칭 실패 개수:", len(missing))
if len(missing) > 0:
  print(missing[["image_path", "parent_name"]].head())


저장 완료: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/251207_segmentation_area_mm2.csv
scale map 매칭 실패 개수: 0


In [ ]:
scale_df.columns

Index(['image_name', 'width_px', 'height_px', 'px_per_mm_x', 'px_per_mm_y',
       'parent_name'],
      dtype='object')

###5. 면적 기반 지표 산출하기

In [ ]:
import os, glob, math, csv
import numpy as np
import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed

IMG_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/251011 양상추 seg 연습(2)/train/images"   # train/valid/test별로 돌려도 됨
LBL_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/251011 양상추 seg 연습(2)/train/labels"   # YOLO-seg txt들
OUT_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/251011 양상추 seg 연습(2)/result/251012_segcalculate(1).csv"
N_WORKERS = 8
TARGET = (640, 640)  # 256x256

def yolo_poly_to_mask(poly_xy, w, h, target=TARGET):
    """정규화 YOLO 세그 좌표 -> 마스크(256x256)"""
    xs = (poly_xy[0::2] * w).astype(np.float32)
    ys = (poly_xy[1::2] * h).astype(np.float32)
    poly = np.stack([xs, ys], axis=1)

    # 원본 크기 기준 빈 마스크
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly.astype(np.int32)], 255)

    # 256x256로 리사이즈 (recentering 포함)
    mask = cv2.resize(mask, target, interpolation=cv2.INTER_NEAREST)
    return mask

def contour_metrics(mask):
    """형상 지표 계산"""
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    perim = cv2.arcLength(cnt, True) + 1e-6

    # 원형도(1이면 완전 원)
    circularity = 4*math.pi*area/(perim*perim)

    # 볼록껍질 대비 지표
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull) + 1e-6
    solidity = area / hull_area
    concavity = (hull_area - area) / hull_area

    # 굴곡(경계 각도 변화량 기반 간단 지표)
    pts = cnt[:,0,:].astype(np.float32)
    dif1 = np.diff(pts, axis=0, prepend=pts[-1:])
    ang = np.arctan2(dif1[:,1], dif1[:,0])
    dtheta = np.diff(ang, prepend=ang[-1:])
    dtheta = (dtheta + np.pi) % (2*np.pi) - np.pi  # unwrap
    curvature_idx = np.mean(np.abs(dtheta))  # 평균 각 변화량

    # 거칠기(roughness): P^2/(4πA) (원형도의 역수)
    roughness = (perim*perim)/(4*math.pi*area) if area>0 else np.nan

    return dict(
        area_px=float(area),
        perim_px=float(perim),
        circularity=float(circularity),
        solidity=float(solidity),
        concavity=float(concavity),
        curvature=float(curvature_idx),
        roughness=float(roughness)
    )

def process_one(txt_path):
    base = os.path.splitext(os.path.basename(txt_path))[0]
    # 이미지 경로 추정 (확장자 다양하면 찾아보기)
    candidates = glob.glob(os.path.join(IMG_DIR, base) + ".*")
    if not candidates:
        return None
    img_path = candidates[0]
    img = cv2.imread(img_path)
    if img is None:
        return None
    h, w = img.shape[:2]

    with open(txt_path, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    # 하나의 이미지에 인스턴스가 여러 개라도, 가장 큰 것 기준으로 지표
    # (필요하면 loop 돌려 합치거나 평균 내도록 수정)
    best = None
    for ln in lines:
        parts = ln.split()
        # cls = int(parts[0])  # 단일 클래스라 무시
        coords = np.array(list(map(float, parts[1:])), dtype=np.float32)
        mask = yolo_poly_to_mask(coords, w, h)
        m = contour_metrics(mask)
        if m is None:
            continue
        if (best is None) or (m["area_px"] > best["area_px"]):
            best = m
    if best is None:
        return None
    best.update(dict(
        filename=os.path.basename(img_path),
        width=w, height=h
    ))
    return best

def main():
    txts = sorted(glob.glob(os.path.join(LBL_DIR, "*.txt")))
    rows = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {ex.submit(process_one, p): p for p in txts}
        for fu in as_completed(futs):
            r = fu.result()
            if r: rows.append(r)

    cols = ["filename","width","height",
            "area_px","perim_px","circularity","solidity","concavity",
            "curvature","roughness"]
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=cols)
        wr.writeheader()
        for r in rows:
            wr.writerow({k: r.get(k, "") for k in cols})

if __name__ == "__main__":
    main()
